# Marúbo (mzr) ↔ Portuguese (por) NLLB-200 600M LoRA Fine-Tuning

This notebook fine-tunes Meta's `facebook/nllb-200-distilled-600M` model for bidirectional translation between **Marúbo** (`mzr_Latn`, Panoan family) and **Portuguese** (`por_Latn`).

### Non-Negotiable Governance & Data Invariants (ADR-0002, AGENTS.md):
1. **Strict Data Lineage**: The training set combines human-attested parallel verses (Costa 2000, ScriptureEarth) with verified, valence-constrained synthetic data.
2. **Zero-Leakage Gold Standard (Invariant 2)**: The test set (`test.hf.jsonl`, 200 verses) consists strictly of 100% human-attested translations; synthetic data is never permitted in evaluation.
3. **Panoan Transfer Learning**: The newly added `mzr_Latn` token is initialized from Shipibo-Konibo (`shp_Latn`, Panoan) embeddings present in NLLB-200.
4. **Community Audit**: Generations are exported to `evaluation_predictions.jsonl` for validation by native speakers.


## 1. Environment Installation
Install compatible versions of HuggingFace transformers, peft, accelerate, datasets, and sacrebleu.

In [ ]:
!pip -q install \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    accelerate>=0.29.0 \
    datasets>=2.18.0 \
    sacrebleu>=2.4.0 \
    bitsandbytes \
    sentencepiece


## 2. Imports & GPU Environment Check

In [ ]:
import os
import json
import torch
import pandas as pd
import sacrebleu
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 3. Configuration & Hyperparameters

In [ ]:
BASE_MODEL = "facebook/nllb-200-distilled-600M"

# NLLB Language Tokens
MZR_LANG = "mzr_Latn"      # Target: Marúbo (Panoan)
PT_LANG = "por_Latn"       # Contact: Portuguese
PANO_SEED_LANG = "shp_Latn" # Transfer source: Shipibo-Konibo (Panoan in NLLB)

# Dataset Paths (exported from LKB pipeline)
TRAIN_PATH = "exports/mzr/datasets/train.hf.jsonl"
VAL_PATH = "exports/mzr/datasets/val.hf.jsonl"
TEST_PATH = "exports/mzr/datasets/test.hf.jsonl"
OUTPUT_DIR = "checkpoints/mzr-nllb-600m-lora"
FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

# Sequence & LoRA Hyperparameters
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Optimization
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
SAVE_STEPS = 500
EVAL_STEPS = 500
LOGGING_STEPS = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


## 4. Load & Validate Dataset Partitions (Strict Invariant 2 Check)

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

raw_train = load_jsonl(TRAIN_PATH)
raw_val = load_jsonl(VAL_PATH)
raw_test = load_jsonl(TEST_PATH)

print(f"Train samples: {len(raw_train)}")
print(f"Val samples:   {len(raw_val)}")
print(f"Test samples:  {len(raw_test)} (100% human gold truth)")

# Governance Invariant 2 assertion
for item in raw_test:
    assert not item.get("synthetic", False), "Invariant 2 Violation: Synthetic row detected in test set!"
print("✓ Invariant 2 verified: Test set is 100% human-attested gold truth.")


## 5. Tokenizer, Special Token & Panoan Transfer Initialization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if tokenizer.convert_tokens_to_ids(MZR_LANG) == tokenizer.unk_token_id:
    tokenizer.add_special_tokens({"additional_special_tokens": [MZR_LANG]})

mzr_token_id = tokenizer.convert_tokens_to_ids(MZR_LANG)
pt_token_id = tokenizer.convert_tokens_to_ids(PT_LANG)
shp_token_id = tokenizer.convert_tokens_to_ids(PANO_SEED_LANG)

print(f"Marúbo token id: {mzr_token_id}")
print(f"Portuguese token id: {pt_token_id}")
print(f"Shipibo-Konibo (Pano seed) token id: {shp_token_id}")

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16 if torch.cuda.is_available() else torch.float32

base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, torch_dtype=dtype)
base_model.resize_token_embeddings(len(tokenizer))

# Transfer embedding weights from related Panoan language (shp) + contact language (por)
with torch.no_grad():
    input_emb = base_model.get_input_embeddings().weight
    output_emb = base_model.get_output_embeddings().weight
    
    seed_ids = [shp_token_id, pt_token_id]
    input_emb[mzr_token_id].copy_(input_emb[seed_ids].mean(dim=0))
    
    if output_emb.data_ptr() != input_emb.data_ptr():
        output_emb[mzr_token_id].copy_(output_emb[seed_ids].mean(dim=0))

base_model.config.use_cache = False

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    trainable_token_indices=[mzr_token_id],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 6. Build Bidirectional Parallel Dataset

In [ ]:
def build_bidirectional_records(raw_items):
    pairs = []
    for r in raw_items:
        trans = r["translation"]
        mzr_text = trans.get("mzr", "").strip()
        por_text = trans.get("por", "").strip()
        if not mzr_text or not por_text:
            continue
        # por -> mzr
        pairs.append({"src_text": por_text, "tgt_text": mzr_text, "src_lang": PT_LANG, "tgt_lang": MZR_LANG, "direction": "por→mzr"})
        # mzr -> por
        pairs.append({"src_text": mzr_text, "tgt_text": por_text, "src_lang": MZR_LANG, "tgt_lang": PT_LANG, "direction": "mzr→por"})
    return pairs

train_pairs = build_bidirectional_records(raw_train)
val_pairs = build_bidirectional_records(raw_val)
test_pairs = build_bidirectional_records(raw_test)

print(f"Bidirectional training samples:   {len(train_pairs)}")
print(f"Bidirectional validation samples: {len(val_pairs)}")
print(f"Bidirectional test samples:       {len(test_pairs)}")

train_ds = Dataset.from_list(train_pairs)
val_ds = Dataset.from_list(val_pairs)
test_ds = Dataset.from_list(test_pairs)


## 7. Tokenization Pipeline

In [ ]:
def preprocess_function(examples):
    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
    
    for src, tgt, s_lang, t_lang in zip(examples["src_text"], examples["tgt_text"], examples["src_lang"], examples["tgt_lang"]):
        tokenizer.src_lang = s_lang
        inp = tokenizer(src, max_length=MAX_SOURCE_LENGTH, truncation=True)
        
        tokenizer.src_lang = t_lang
        out = tokenizer(text_target=tgt, max_length=MAX_TARGET_LENGTH, truncation=True)
        
        model_inputs["input_ids"].append(inp["input_ids"])
        model_inputs["attention_mask"].append(inp["attention_mask"])
        model_inputs["labels"].append(out["input_ids"])
        
    return model_inputs

tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
print("Tokenization complete!")


## 8. Training Setup & Execution

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=LOGGING_STEPS,
    predict_with_generate=True,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    report_to=["none"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

print(f"Saving final LoRA adapter to {FINAL_ADAPTER_DIR}...")
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("Model saved successfully!")


## 9. Evaluation on Gold-Standard Test Set (chrF++ & BLEU)

In [ ]:
@torch.inference_mode()
def evaluate_test_direction(dataset, direction, batch_size=8):
    filtered = [item for item in dataset if item["direction"] == direction]
    srcs = [item["src_text"] for item in filtered]
    refs = [item["tgt_text"] for item in filtered]
    s_lang = filtered[0]["src_lang"]
    t_lang = filtered[0]["tgt_lang"]
    
    preds = []
    model.eval()
    forced_bos = tokenizer.convert_tokens_to_ids(t_lang)
    
    for i in range(0, len(srcs), batch_size):
        b_srcs = srcs[i:i+batch_size]
        tokenizer.src_lang = s_lang
        enc = tokenizer(b_srcs, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LENGTH).to(model.device)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_new_tokens=128, num_beams=4)
        dec = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(dec)
        
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    
    return {"direction": direction, "samples": len(refs), "chrF++": chrf, "BLEU": bleu}, srcs, preds, refs

results = []
predictions_records = []

for dir_name in ["por→mzr", "mzr→por"]:
    metrics, s_list, p_list, r_list = evaluate_test_direction(test_pairs, dir_name)
    results.append(metrics)
    for s, p, r in zip(s_list, p_list, r_list):
        predictions_records.append({"direction": dir_name, "source": s, "prediction": p, "reference": r})

df_results = pd.DataFrame(results)
display(df_results)

# Save predictions for community elder / speaker review
preds_path = os.path.join(FINAL_ADAPTER_DIR, "evaluation_predictions.jsonl")
with open(preds_path, "w", encoding="utf-8") as f:
    for rec in predictions_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Saved {len(predictions_records)} predictions for community review to {preds_path}.")


## 10. Interactive Translation Inference Playground

In [ ]:
def translate(text, src_lang=PT_LANG, tgt_lang=MZR_LANG):
    tokenizer.src_lang = src_lang
    forced_bos = tokenizer.convert_tokens_to_ids(tgt_lang)
    enc = tokenizer([text], return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_new_tokens=128, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample_texts = [
    "O menino vê o peixe no rio.",
    "A mulher não come a carne.",
    "Nós caminhamos na aldeia."
]

print("--- Sample Translations (Portuguese ➔ Marúbo) ---")
for s in sample_texts:
    print(f"PT:  {s}")
    print(f"MZR: {translate(s, src_lang=PT_LANG, tgt_lang=MZR_LANG)}\n")
